In [ ]:
# imports
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torchvision.transforms.v2 import RandomHorizontalFlip, RandomVerticalFlip, RandomRotation
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, Dataset, DataLoader
import torch.optim as optim
import numpy as np
from torchsummary import summary
from transformers import AutoImageProcessor, EfficientNetConfig, EfficientNetForImageClassification

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

/opt/prak/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Этап 1. Загрузка и предобработка данных

processor = AutoImageProcessor.from_pretrained('google/efficientnet-b2')
processor

train_dataset = ImageFolder('data/ogyeiv2/ogyeiv2/train', transform=processor)
val_dataset = ImageFolder('data/ogyeiv2/ogyeiv2/test', transform=processor)

train_loader = DataLoader(train_dataset, shuffle=True, batch_size=32)
val_loader = DataLoader(val_dataset, shuffle=False, batch_size=64)

# Проверка
print("Количество изображений в train:", len(train_dataset))
print("Количество изображений в val:", len(val_dataset))
print("Список классов:", train_dataset.classes) 
print("Количество классов:", len(train_dataset.classes))

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Количество изображений в train: 2352
Количество изображений в val: 504
Список классов: ['acc_long_600_mg', 'advil_ultra_forte', 'akineton_2_mg', 'algoflex_forte_dolo_400_mg', 'algoflex_rapid_400_mg', 'algopyrin_500_mg', 'ambroxol_egis_30_mg', 'apranax_550_mg', 'aspirin_ultra_500_mg', 'atoris_20_mg', 'atorvastatin_teva_20_mg', 'betaloc_50_mg', 'bila_git', 'c_vitamin_teva_500_mg', 'calci_kid', 'cataflam_50_mg', 'cataflam_dolo_25_mg', 'cetirizin_10_mg', 'cold_fx', 'coldrex', 'concor_10_mg', 'concor_5_mg', 'condrosulf_800_mg', 'controloc_20_mg', 'covercard_plus_10_mg_2_5_mg_5_mg', 'coverex_4_mg', 'diclopram_75-mg_20-mg', 'dorithricin_mentol', 'dulsevia_60_mg', 'enterol_250_mg', 'favipiravir_meditop_200_mg', 'ibumax_400_mg', 'jutavit_c_vitamin', 'jutavit_cink', 'kalcium_magnezium_cink', 'kalium_r', 'koleszterin_kontroll', 'lactamed', 'lactiv_plus', 'laresin_10_mg', 'letrox_50_mikrogramm', 'lordestin_5_mg', 'merckformin_xr_1000_mg', 'meridian', 'metothyrin_10_mg', 'mezym_forte_10_000_egyseg'

In [3]:
# Этап 2. Объявление модели
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = EfficientNetConfig().from_pretrained('google/efficientnet-b2')
model = EfficientNetForImageClassification(config).from_pretrained('google/efficientnet-b2')
model.classifier = nn.Linear(in_features=1408, out_features=84, bias=True)
model.to(device)
for p in model.parameters():
    p.requires_grad = False
for i, p in enumerate(model.efficientnet.encoder.blocks[22].parameters()):
    p.requires_grad = True
for p in model.classifier.parameters():
    p.requires_grad = True

print(model)
# Проверка
#summary(model, input_size=(3, 224, 224), device='cuda') 

EfficientNetForImageClassification(
  (efficientnet): EfficientNetModel(
    (embeddings): EfficientNetEmbeddings(
      (padding): ZeroPad2d((0, 1, 0, 1))
      (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=valid, bias=False)
      (batchnorm): BatchNorm2d(32, eps=0.001, momentum=0.99, affine=True, bias=True, track_running_stats=True)
      (activation): SiLU()
    )
    (encoder): EfficientNetEncoder(
      (blocks): ModuleList(
        (0): EfficientNetBlock(
          (depthwise_conv): EfficientNetDepthwiseLayer(
            (depthwise_conv_pad): ZeroPad2d((0, 1, 0, 1))
            (depthwise_conv): EfficientNetDepthwiseConv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same, groups=32, bias=False)
            (depthwise_norm): BatchNorm2d(32, eps=0.001, momentum=0.99, affine=True, bias=True, track_running_stats=True)
            (depthwise_act): SiLU()
          )
          (squeeze_excite): EfficientNetSqueezeExciteLayer(
            (squeeze): Ad

In [11]:
# Этап 3. Обучение или дообучение

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print(torch.cuda.get_device_name(0))
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3,)
criterion = nn.CrossEntropyLoss()
epochs = 10
sheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2,gamma=0.4)
best_vloss = 1e5

ls_train, ls_test = [], []

for epoch in range(epochs):
    print(f'Эпоха {epoch}')
    loss_epoch_train = []
    model.train()
    for idx, data in enumerate(train_loader):
        bucket_image, label = data
        image = bucket_image['pixel_values'][0].to(device)

        y_pred = model(image).logits
        loss = criterion(y_pred, label.to(device))
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        loss_epoch_train.append(loss.detach().item())
    sheduler.step()
    mean_loss_epoch_train = np.mean(loss_epoch_train)
    ls_train.append(mean_loss_epoch_train)

    loss_epoch_test = []
    model.eval()
    for idx, data in enumerate(val_loader):
        bucket_image, label = data
        image = bucket_image['pixel_values'][0].to(device)
        y_pred = model(image).logits
        loss = criterion(y_pred, label.to(device))
        loss_epoch_test.append(loss.detach().item())
    mean_loss_epoch_test = np.mean(loss_epoch_test)
    ls_test.append(mean_loss_epoch_test)    

    # Сохранение лучшей модели
    if mean_loss_epoch_test < best_vloss:
        best_vloss = mean_loss_epoch_test
        model_path = f'pills_classifier_{epoch}.pt'
        torch.save(model.state_dict(), model_path)
        print(f'Сохранен чекпойнт pills_classifier_{epoch}.pt')
    print(f'В конце эпохи ошибка train {mean_loss_epoch_train}, ошибка val {mean_loss_epoch_test}')

Device: cuda
NVIDIA GeForce RTX 5090
Эпоха 0
Сохранен чекпойнт pills_classifier_0.pt
В конце эпохи ошибка train 0.4223400803433882, ошибка val 0.5128134973347187
Эпоха 1
Сохранен чекпойнт pills_classifier_1.pt
В конце эпохи ошибка train 0.15701661258935928, ошибка val 0.36753261648118496
Эпоха 2
Сохранен чекпойнт pills_classifier_2.pt
В конце эпохи ошибка train 0.08509286749805953, ошибка val 0.33315866626799107
Эпоха 3
Сохранен чекпойнт pills_classifier_3.pt
В конце эпохи ошибка train 0.05103310190040518, ошибка val 0.20857654325664043
Эпоха 4
В конце эпохи ошибка train 0.041753756612337926, ошибка val 0.4353455789387226
Эпоха 5
В конце эпохи ошибка train 0.037115857219071804, ошибка val 0.2501373505219817
Эпоха 6
В конце эпохи ошибка train 0.03212954307830817, ошибка val 0.24220941681414843
Эпоха 7
В конце эпохи ошибка train 0.03359239191018246, ошибка val 0.236271096393466
Эпоха 8
В конце эпохи ошибка train 0.028093639610184205, ошибка val 0.2814777661114931
Эпоха 9
В конце эпохи ош

In [ ]:
# 4. Оценка качества

state_dict = torch.load('pills_classifier_3.pt', map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()


def extract_images(batch):
    """
    Достаёт tensor изображений из batch.
    Нужно из-за AutoImageProcessor, который может вернуть dict / BatchFeature / list.
    """

    if isinstance(batch, dict) or hasattr(batch, "keys"):
        x = batch["pixel_values"]
    else:
        x = batch

    if isinstance(x, list):
        if len(x) == 1 and torch.is_tensor(x[0]):
            x = x[0]
        elif all(torch.is_tensor(item) for item in x):
            x = torch.stack(x)
        else:
            x = torch.tensor(np.array(x))

    if not torch.is_tensor(x):
        x = torch.tensor(x)

    # Убираем лишнюю размерность: [B, 1, C, H, W] -> [B, C, H, W]
    if x.dim() == 5 and x.shape[1] == 1:
        x = x.squeeze(1)

    return x.float()


y_true = []
y_pred = []

val_eval_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

with torch.no_grad():
    for batch_images, labels in val_eval_loader:
        images = extract_images(batch_images).to(device)
        labels = labels.to(device)

        outputs = model(images)

        if hasattr(outputs, "logits"):
            logits = outputs.logits
        else:
            logits = outputs

        preds = logits.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())


y_true = np.array(y_true)
y_pred = np.array(y_pred)

classes = val_dataset.classes
class_ids = list(range(len(classes)))

accuracy = accuracy_score(y_true, y_pred)

print("=" * 80)
print(f"Общая accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")

if accuracy > 0.75:
    print("Требование accuracy выше 75% выполнено.")
else:
    print("Требование accuracy выше 75% НЕ выполнено.")

print("=" * 80)
print("Precision, Recall, F1 по каждому классу:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=class_ids,
        target_names=classes,
        digits=4,
        zero_division=0
    )
)

cm = confusion_matrix(y_true, y_pred, labels=class_ids)

support_by_class = cm.sum(axis=1)
correct_by_class = np.diag(cm)
errors_by_class = support_by_class - correct_by_class

false_negative_by_class = cm.sum(axis=1) - np.diag(cm)
false_positive_by_class = cm.sum(axis=0) - np.diag(cm)

worst_classes = [
    i for i in np.argsort(errors_by_class)[::-1]
    if errors_by_class[i] > 0
][:5]

print("=" * 80)
print("На каких 5 классах модель ошибается чаще всего:")

if len(worst_classes) == 0:
    print("Ошибок на валидации не найдено.")
else:
    for class_idx in worst_classes:
        row = cm[class_idx].copy()
        row[class_idx] = 0

        confused_with = [
            j for j in np.argsort(row)[::-1]
            if row[j] > 0
        ][:3]

        confused_text = ", ".join(
            f"{classes[j]} ({row[j]} раз)"
            for j in confused_with
        )

        if not confused_text:
            confused_text = "нет устойчивого класса-пары"

        print(
            f"- {classes[class_idx]}: "
            f"ошибок {errors_by_class[class_idx]} из {support_by_class[class_idx]}, "
            f"правильно {correct_by_class[class_idx]}; "
            f"чаще всего путается с: {confused_text}"
        )


perfect_classes = [
    i for i in class_ids
    if support_by_class[i] > 0
    and false_negative_by_class[i] == 0
    and false_positive_by_class[i] == 0
]

print("=" * 80)
print("На каких классах модель не совершает ошибок:")

if len(perfect_classes) == 0:
    print("По строгому критерию таких классов нет.")
else:
    for class_idx in perfect_classes:
        print(f"- {classes[class_idx]}: {support_by_class[class_idx]} изображений, ошибок 0")


print("=" * 80)
print("Ответы на вопросы:")

print("\n1. Почему модель может ошибаться на этих классах?")
print(
    "Очень похожие изображения при дисбалансе между классами, " 
    "в следствие чего сеть из двух одинаково подходящих вариантов выбирает наиболее популярный."    
)

print("\n2. Почему некоторые классы модель распознаёт безошибочно?")
print(
    "Наличие уникальных ноборов признаков помогает точной классификации. "
    "Так же выборки val и train содержат достаточно похожие изображения "
)

print("\n3. Как можно улучшить точность классификатора?")
print(
    "- Добавить больше изображений для классов, где больше всего ошибок.\n"
    "- Добавить аугментации: повороты, изменение яркости, контраста, blur, crop.\n"
    "- Проверить разметку: возможно, часть изображений лежит не в тех папках.\n"
    "- Увеличить число эпох или уменьшить learning rate.\n"
    "- Использовать balanced sampler или class weights, если классы несбалансированы.\n"
    "- Попробовать более сильную модель или fine-tuning большего числа слоёв."
)

print("\n4. Как ещё можно проанализировать результаты и ошибки модели?")
print(
    "- Построить confusion matrix.\n"
    "- Посмотреть top-ошибки с картинками: настоящий класс, предсказанный класс, confidence.\n"
    "- Найти классы с низким recall: модель часто не узнаёт этот класс.\n"
    "- Найти классы с низким precision: модель слишком часто ошибочно предсказывает этот класс.\n"
    "- Проверить confidence модели: где она ошибается уверенно, а где сомневается.\n"
    "- Посмотреть ошибки по условиям съёмки: ракурс, свет, фон, обрезка, качество фото."
)

Общая accuracy: 0.9444 (94.44%)
Требование accuracy выше 75% выполнено.
Precision, Recall, F1 по каждому классу:
                                  precision    recall  f1-score   support

                 acc_long_600_mg     1.0000    1.0000    1.0000         6
               advil_ultra_forte     0.8571    1.0000    0.9231         6
                   akineton_2_mg     1.0000    1.0000    1.0000         6
      algoflex_forte_dolo_400_mg     1.0000    1.0000    1.0000         6
           algoflex_rapid_400_mg     1.0000    0.8333    0.9091         6
                algopyrin_500_mg     1.0000    1.0000    1.0000         6
             ambroxol_egis_30_mg     1.0000    1.0000    1.0000         6
                  apranax_550_mg     1.0000    1.0000    1.0000         6
            aspirin_ultra_500_mg     0.8333    0.8333    0.8333         6
                    atoris_20_mg     0.8571    1.0000    0.9231         6
         atorvastatin_teva_20_mg     1.0000    0.8333    0.9091         